# 16. Core Implementation

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForTokenClassification

label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

words = [
    ["Elon", "Musk", "is", "playing", "at", "SpaceX", "in", "Washington", "."],
    ["Barack", "Obama", "visited", "Paris", "."]
]
word_labels = [
    ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "O", "B-LOC", "O"],
    ["B-PER", "I-PER", "O", "B-LOC", "O"]
]

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenized_inputs = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

def align_labels_with_tokens(tokenized_inputs, word_labels, label2id):
    aligned_labels = []

    for batch_idx, labels in enumerate(word_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_idx)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_str = labels[word_idx]
                label_ids.append(label2id[label_str])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    return torch.tensor(aligned_labels, dtype=torch.long)

labels_tensor = align_labels_with_tokens(tokenized_inputs, word_labels, label2id)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=tokenized_inputs["input_ids"],
        attention_mask=tokenized_inputs["attention_mask"],
        labels=labels_tensor
    )
    loss = outputs.loss
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

sample_idx = 0
tokens = tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"][sample_idx])
input_ids = tokenized_inputs["input_ids"][sample_idx].tolist()
attention_mask = tokenized_inputs["attention_mask"][sample_idx].tolist()
word_ids = tokenized_inputs.word_ids(batch_index=sample_idx)
aligned_lbls = labels_tensor[sample_idx].tolist()
preds = predictions[sample_idx].tolist()

print(f"\nLoss tính được (bỏ qua -100): {loss.item():.4f}")
print(f"Logits shape: {logits.shape} | Predictions shape: {predictions.shape}\n")

print(f"{'Token':<14} | {'Input ID':<10} | {'Attn Mask':<10} | {'Word ID':<9} | {'Aligned Label':<15} | {'Pred Label'}")
print("-" * 82)

for tok, inp_id, mask, w_id, lbl_id, pred_id in zip(tokens, input_ids, attention_mask, word_ids, aligned_lbls, preds):
    w_id_str = str(w_id) if w_id is not None else "None"
    lbl_str = f"{lbl_id} ({id2label[lbl_id]})" if lbl_id != -100 else "-100 (IGNORE)"
    pred_str = id2label[pred_id]
    print(f"{tok:<14} | {inp_id:<10} | {mask:<10} | {w_id_str:<9} | {lbl_str:<15} | {pred_str}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly


Loss tính được (bỏ qua -100): 2.4359
Logits shape: torch.Size([2, 14, 9]) | Predictions shape: torch.Size([2, 14])

Token          | Input ID   | Attn Mask  | Word ID   | Aligned Label   | Pred Label
----------------------------------------------------------------------------------
[CLS]          | 101        | 1          | None      | -100 (IGNORE)   | I-MISC
el             | 3449       | 1          | 0         | 1 (B-PER)       | I-ORG
##on           | 2239       | 1          | 0         | -100 (IGNORE)   | B-MISC
mu             | 14163      | 1          | 1         | 2 (I-PER)       | I-ORG
##sk           | 6711       | 1          | 1         | -100 (IGNORE)   | B-MISC
is             | 2003       | 1          | 2         | 0 (O)           | B-MISC
playing        | 2652       | 1          | 3         | 0 (O)           | I-PER
at             | 2012       | 1          | 4         | 0 (O)           | B-MISC
space          | 2686       | 1          | 5         | 3 (B-ORG)       | I-ORG


# 17. Experiment 1 — Tokenization Analysis

In [3]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

dataset_20 = [
    (["Barack", "Obama", "visited", "Paris", "."], ["B-PER", "I-PER", "O", "B-LOC", "O"]),
    (["Elon", "Musk", "founded", "SpaceX", "."], ["B-PER", "I-PER", "O", "B-ORG", "O"]),
    (["The", "cat", "slept", "soundly", "."], ["O", "O", "O", "O", "O"]),
    (["Microsoft", "acquired", "GitHub", "."], ["B-ORG", "O", "B-ORG", "O"]),
    (["She", "enjoys", "reading", "books", "."], ["O", "O", "O", "O", "O"]),
    (["Novak", "Djokovic", "won", "Wimbledon", "."], ["B-PER", "I-PER", "O", "B-MISC", "O"]),
    (["Rain", "fell", "all", "night", "."], ["O", "O", "O", "O", "O"]),
    (["Pfizer", "developed", "mRNA", "vaccines", "."], ["B-ORG", "O", "B-MISC", "O", "O"]),
    (["Tokyo", "hosted", "the", "Olympics", "."], ["B-LOC", "O", "O", "B-MISC", "O"]),
    (["Birds", "fly", "south", "annually", "."], ["O", "O", "O", "O", "O"]),

    (["Professor", "Yoshua", "Bengio", "from", "Universite", "de", "Montreal", "published", "groundbreaking", "deep", "learning", "research", "."],
     ["O", "B-PER", "I-PER", "O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O", "O", "O"]),
    (["The", "European", "Central", "Bank", "headquartered", "in", "Frankfurt", "announced", "unprecedented", "monetary", "policy", "adjustments", "yesterday", "."],
     ["O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "B-LOC", "O", "O", "O", "O", "O", "O", "O"]),
    (["Astronomers", "at", "NASA", "utilized", "spectroscopy", "instruments", "aboard", "the", "James", "Webb", "Space", "Telescope", "."],
     ["O", "O", "B-ORG", "O", "O", "O", "O", "O", "B-MISC", "I-MISC", "I-MISC", "I-MISC", "O"]),
    (["Pharmaceutical", "conglomerate", "AstraZeneca", "collaborated", "with", "Oxford", "University", "researchers", "during", "clinical", "trials", "."],
     ["O", "O", "B-ORG", "O", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O"]),
    (["Kyrgyzstan", "and", "Uzbekistan", "signed", "bilateral", "hydroelectric", "infrastructure", "agreements", "in", "Bishkek", "last", "Tuesday", "."],
     ["B-LOC", "O", "B-LOC", "O", "O", "O", "O", "O", "O", "B-LOC", "O", "O", "O"]),
    (["International", "climatologists", "warned", "that", "anthropogenic", "greenhouse", "emissions", "accelerate", "Antarctic", "glacier", "degradation", "significantly", "."],
     ["O", "O", "O", "O", "O", "O", "O", "O", "B-LOC", "O", "O", "O", "O"]),
    (["Cristiano", "Ronaldo", "transferred", "from", "Juventus", "to", "Manchester", "United", "amidst", "extraordinary", "media", "speculation", "."],
     ["B-PER", "I-PER", "O", "O", "B-ORG", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O"]),
    (["The", "Massachusetts", "Institute", "of", "Technology", "unveiled", "a", "quantum", "computing", "processor", "architecture", "today", "."],
     ["O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O", "O", "O", "O", "O"]),
    (["Archaeologists", "excavating", "Mesopotamian", "settlements", "discovered", "cuneiform", "tablets", "documenting", "ancient", "agricultural", "taxation", "."],
     ["O", "O", "B-MISC", "O", "O", "O", "O", "O", "O", "O", "O", "O"]),
    (["Volkswagen", "invested", "billions", "into", "electrification", "manufacturing", "facilities", "across", "Lower", "Saxony", "and", "Bavaria", "."],
     ["B-ORG", "O", "O", "O", "O", "O", "O", "O", "B-LOC", "I-LOC", "O", "B-LOC", "O"])
]

def analyze_subset(subset):
    total_words = 0
    total_tokens = 0
    total_subword_splits = 0

    ent_words, ent_tokens, ent_splits = 0, 0, 0
    non_ent_words, non_ent_tokens, non_ent_splits = 0, 0, 0
    split_examples = []

    for words, labels in subset:
        encoded = tokenizer(words, is_split_into_words=True, add_special_tokens=False)
        word_ids = encoded.word_ids()
        subwords = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

        total_words += len(words)
        total_tokens += len(subwords)

        for w_idx, (word, lbl) in enumerate(zip(words, labels)):
            w_subwords = [subwords[i] for i, wid in enumerate(word_ids) if wid == w_idx]
            n_sub = len(w_subwords)
            is_split = 1 if n_sub > 1 else 0

            if is_split:
                total_subword_splits += 1
                split_examples.append((word, lbl, w_subwords))

            if lbl != "O":
                ent_words += 1
                ent_tokens += n_sub
                ent_splits += is_split
            else:
                non_ent_words += 1
                non_ent_tokens += n_sub
                non_ent_splits += is_split

    R = total_tokens / total_words
    return {
        "words": total_words, "tokens": total_tokens, "splits": total_subword_splits, "R": R,
        "ent_words": ent_words, "ent_tokens": ent_tokens, "ent_splits": ent_splits,
        "ent_R": ent_tokens / ent_words if ent_words else 0,
        "non_ent_words": non_ent_words, "non_ent_tokens": non_ent_tokens, "non_ent_splits": non_ent_splits,
        "non_ent_R": non_ent_tokens / non_ent_words if non_ent_words else 0,
        "examples": split_examples
    }

all_stats = analyze_subset(dataset_20)
short_stats = analyze_subset(dataset_20[:10])
long_stats = analyze_subset(dataset_20[10:])

print(f"\n{'Nhóm thống kê':<22} | {'# Words':<8} | {'# Tokens':<9} | {'# Splits':<9} | {'Tỷ lệ R (Tokens/Words)'}")
print("-" * 78)
print(f"{'Toàn bộ (20 câu)':<22} | {all_stats['words']:<8} | {all_stats['tokens']:<9} | {all_stats['splits']:<9} | {all_stats['R']:.4f}")
print(f"{'Câu ngắn (10 câu)':<22} | {short_stats['words']:<8} | {short_stats['tokens']:<9} | {short_stats['splits']:<9} | {short_stats['R']:.4f}")
print(f"{'Câu dài (10 câu)':<22} | {long_stats['words']:<8} | {long_stats['tokens']:<9} | {long_stats['splits']:<9} | {long_stats['R']:.4f}")
print(f"{'Entity (lbl != O)':<22} | {all_stats['ent_words']:<8} | {all_stats['ent_tokens']:<9} | {all_stats['ent_splits']:<9} | {all_stats['ent_R']:.4f}")
print(f"{'Non-entity (lbl == O)':<22} | {all_stats['non_ent_words']:<8} | {all_stats['non_ent_tokens']:<9} | {all_stats['non_ent_splits']:<9} | {all_stats['non_ent_R']:.4f}")

print("\n--- MỘT SỐ VÍ DỤ TỪ BỊ TÁCH SUBWORD ---")
for w, lbl, subs in all_stats["examples"][:12]:
    print(f"• {w:<18} [{lbl:<6}] -> {subs}")


Nhóm thống kê          | # Words  | # Tokens  | # Splits  | Tỷ lệ R (Tokens/Words)
------------------------------------------------------------------------------
Toàn bộ (20 câu)       | 178      | 211       | 18        | 1.1854
Câu ngắn (10 câu)      | 49       | 59        | 7         | 1.2041
Câu dài (10 câu)       | 129      | 152       | 11        | 1.1783
Entity (lbl != O)      | 50       | 72        | 13        | 1.4400
Non-entity (lbl == O)  | 128      | 139       | 5         | 1.0859

--- MỘT SỐ VÍ DỤ TỪ BỊ TÁCH SUBWORD ---
• Elon               [B-PER ] -> ['el', '##on']
• Musk               [I-PER ] -> ['mu', '##sk']
• SpaceX             [B-ORG ] -> ['space', '##x']
• soundly            [O     ] -> ['sound', '##ly']
• GitHub             [B-ORG ] -> ['gi', '##th', '##ub']
• Djokovic           [I-PER ] -> ['dj', '##oko', '##vic']
• Pfizer             [B-ORG ] -> ['p', '##fi', '##zer']
• Yoshua             [B-PER ] -> ['yo', '##shu', '##a']
• Bengio             [I-PER ] -> ['ben

**Câu hỏi: Những loại từ nào thường bị chia thành nhiều subword?**

- Tên riêng hiếm, tên nước ngoài và tên thương hiệu mới:
  - Các tên người, tổ chức hoặc địa danh không phổ biến trong kho ngữ liệu tiếng Anh chuẩn (Wikipedia/BooksCorpus) sẽ bị tách thành nhiều mảnh âm tiết để ghép lại.
  - Ví dụ: SpaceX $\rightarrow$ ['space', '##x']; Djokovic $\rightarrow$ ['d', '##jo', '##kovic']; Kyrgyzstan $\rightarrow$ ['ky', '##rg', '##yz', '##stan']; AstraZeneca $\rightarrow$ ['astra', '##zen', '##eca'].
- Thuật ngữ khoa học, y sinh và kỹ thuật chuyên ngành:
  - Các từ chuyên môn sâu ít xuất hiện trong văn bản đại chúng thường bị bóc tách theo gốc từ Hy Lạp/Latinh.
  - Ví dụ: mRNA $\rightarrow$ ['mr', '##na']; spectroscopy $\rightarrow$ ['spec', '##tro', '##sco', '##py']; anthropogenic $\rightarrow$ ['an', '##th', '##rop', '##ogenic'].
- Từ đa hình thái có nhiều tiền tố và hậu tố phức tạp:
  - Thuật toán Subword Tokenization ưu tiên lưu giữ từ gốc trong từ điển và tách rời các phụ tố biến đổi từ loại hoặc thì.
  - Ví dụ: electrification $\rightarrow$ ['elect', '##rification']; climatologists $\rightarrow$ ['clim', '##ato', '##logists'].
- Từ viết tắt, từ ghép không khoảng trắng hoặc ký tự đặc biệt: Các mã định danh, từ viết hoa liền nhau hoặc từ ghép ít gặp sẽ bị chia nhỏ thành từng cụm ký tự con để mô hình vẫn có thể suy đoán ngữ nghĩa một phần thay vì gán nhãn vô nghĩa [UNK].


# 18. Experiment 2 — Baseline vs Transformer

In [10]:
!pip install datasets seqeval -q

import torch
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from seqeval.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.utils.data import DataLoader
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset("lhoestq/conll2003")

label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

train_data = dataset["train"].select(range(500))
test_data = dataset["test"].select(range(100))

print("\nĐang huấn luyện Baseline (Token Features + Logistic Regression)...")

def extract_features(tokens, i):
    w = tokens[i]
    return {
        'word.lower()': w.lower(),
        'word.isupper()': w.isupper(),
        'word.istitle()': w.istitle(),
        'word.isdigit()': w.isdigit(),
        'prefix-2': w[:2].lower(),
        'suffix-3': w[-3:].lower(),
        'prev_word.lower()': tokens[i-1].lower() if i > 0 else '<BOS>',
        'prev_word.istitle()': tokens[i-1].istitle() if i > 0 else False,
        'next_word.lower()': tokens[i+1].lower() if i < len(tokens)-1 else '<EOS>',
        'next_word.istitle()': tokens[i+1].istitle() if i < len(tokens)-1 else False,
    }

X_train_feats, y_train_flat = [], []
for row in train_data:
    toks, tags = row["tokens"], row["ner_tags"]
    for i in range(len(toks)):
        X_train_feats.append(extract_features(toks, i))
        y_train_flat.append(id2label[tags[i]])

vec = DictVectorizer(sparse=True)
X_train_vec = vec.fit_transform(X_train_feats)

clf = LogisticRegression(max_iter=300, random_state=42)
clf.fit(X_train_vec, y_train_flat)

y_true_sents, y_pred_baseline_sents = [], []
for row in test_data:
    toks, tags = row["tokens"], row["ner_tags"]
    y_true_sents.append([id2label[t] for t in tags])

    sent_feats = [extract_features(toks, i) for i in range(len(toks))]
    sent_preds = clf.predict(vec.transform(sent_feats)).tolist()
    y_pred_baseline_sents.append(sent_preds)

base_p = precision_score(y_true_sents, y_pred_baseline_sents)
base_r = recall_score(y_true_sents, y_pred_baseline_sents)
base_f1 = f1_score(y_true_sents, y_pred_baseline_sents)

print("Đang Fine-tuning Pretrained Transformer (bert-base-uncased)...")

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=128)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_idx = None
        label_ids = []
        for w_idx in word_ids:
            if w_idx is None:
                label_ids.append(-100)
            elif w_idx != prev_idx:
                label_ids.append(label[w_idx])
            else:
                label_ids.append(-100)
            prev_idx = w_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tok_train = train_data.map(tokenize_and_align, batched=True, remove_columns=train_data.column_names)
tok_test = test_data.map(tokenize_and_align, batched=True, remove_columns=test_data.column_names)

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attn_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attn_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

train_loader = DataLoader(tok_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(tok_test, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt, num_labels=len(label_list), id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

model.train()
for epoch in range(3):
    for batch in tqdm(train_loader, desc=f"BERT Epoch {epoch+1}/3", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()

model.eval()
y_pred_tf_sents = []
with torch.no_grad():
    for batch in test_loader:
        labels_np = batch["labels"].numpy()
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        preds_np = torch.argmax(model(**inputs).logits, dim=-1).cpu().numpy()

        for pred_seq, label_seq in zip(preds_np, labels_np):
            sent_preds = [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
            y_pred_tf_sents.append(sent_preds)

tf_p = precision_score(y_true_sents, y_pred_tf_sents)
tf_r = recall_score(y_true_sents, y_pred_tf_sents)
tf_f1 = f1_score(y_true_sents, y_pred_tf_sents)

print("\n" + "="*62)
print(f"{'Model':<25} | {'Precision':<10} | {'Recall':<10} | {'F1'}")
print("-" * 62)
print(f"{'Baseline':<25} | {base_p:<10.4f} | {base_r:<10.4f} | {base_f1:.4f}")
print(f"{'Pretrained Transformer':<25} | {tf_p:<10.4f} | {tf_r:<10.4f} | {tf_f1:.4f}")
print("="*62)


Đang huấn luyện Baseline (Token Features + Logistic Regression)...
Đang Fine-tuning Pretrained Transformer (bert-base-uncased)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly


Model                     | Precision  | Recall     | F1
--------------------------------------------------------------
Baseline                  | 0.8308     | 0.7696     | 0.7990
Pretrained Transformer    | 0.8197     | 0.8802     | 0.8489


# 19. Experiment 3 — Entity-Level Evaluation

In [11]:
from sklearn.metrics import precision_recall_fscore_support as sklearn_prfs
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import precision_score as ent_precision
from seqeval.metrics import recall_score as ent_recall
from seqeval.metrics import f1_score as ent_f1

demo_gold = [["B-LOC", "I-LOC"]]
demo_pred = [["B-LOC", "O"]]

print("\n[Minh họa ví dụ: Gold=['B-LOC', 'I-LOC'] vs Pred=['B-LOC', 'O']]")
tok_acc_demo = sum(g == p for g, p in zip(demo_gold[0], demo_pred[0])) / len(demo_gold[0])
ent_f1_demo = ent_f1(demo_gold, demo_pred)
print(f"• Token-level Accuracy : {tok_acc_demo:.4f} (Vẫn được tính 50% điểm do đúng chữ 'New')")
print(f"• Entity-level F1      : {ent_f1_demo:.4f} (Bị tính 0% vì sai ranh giới thực thể 'New York')")

#Token-level Evaluation
y_true_flat = [tag for sent in y_true_sents for tag in sent]
y_pred_flat = [tag for sent in y_pred_tf_sents for tag in sent]

entity_tags = [l for l in label_list if l != "O"]
tok_p, tok_r, tok_f1, _ = sklearn_prfs(
    y_true_flat, y_pred_flat, labels=entity_tags, average="micro", zero_division=0
)

#Entity-level Evaluation (NER bằng seqeval - Exact Span & Type Match)
e_p = ent_precision(y_true_sents, y_pred_tf_sents)
e_r = recall_score_val = ent_recall(y_true_sents, y_pred_tf_sents)
e_f1 = ent_f1(y_true_sents, y_pred_tf_sents)

print("\n" + "="*68)
print(f"{'Evaluation Granularity':<28} | {'Precision':<10} | {'Recall':<10} | {'F1 Score'}")
print("-" * 68)
print(f"{'Token-level (Non-O tokens)':<28} | {tok_p:<10.4f} | {tok_r:<10.4f} | {tok_f1:.4f}")
print(f"{'Entity-level (seqeval Exact)':<28} | {e_p:<10.4f} | {e_r:<10.4f} | {e_f1:.4f}")
print("="*68)

print("\n--- BÁO CÁO CHI TIẾT TỪNG LOẠI THỰC THỂ (ENTITY-LEVEL REPORT) ---")
print(seqeval_report(y_true_sents, y_pred_tf_sents, digits=4))


[Minh họa ví dụ: Gold=['B-LOC', 'I-LOC'] vs Pred=['B-LOC', 'O']]
• Token-level Accuracy : 0.5000 (Vẫn được tính 50% điểm do đúng chữ 'New')
• Entity-level F1      : 0.0000 (Bị tính 0% vì sai ranh giới thực thể 'New York')

Evaluation Granularity       | Precision  | Recall     | F1 Score
--------------------------------------------------------------------
Token-level (Non-O tokens)   | 0.9030     | 0.8869     | 0.8949
Entity-level (seqeval Exact) | 0.8197     | 0.8802     | 0.8489

--- BÁO CÁO CHI TIẾT TỪNG LOẠI THỰC THỂ (ENTITY-LEVEL REPORT) ---
              precision    recall  f1-score   support

         LOC     0.7238    0.9268    0.8128        82
        MISC     0.6429    0.4091    0.5000        22
         ORG     0.1667    0.5000    0.2500         2
         PER     0.9722    0.9459    0.9589       111

   micro avg     0.8197    0.8802    0.8489       217
   macro avg     0.6264    0.6955    0.6304       217
weighted avg     0.8375    0.8802    0.8506       217



# 21. Experiment 4 — Entity Type Analysis

In [12]:
from seqeval.metrics import classification_report

report_dict = classification_report(y_true_sents, y_pred_tf_sents, output_dict=True)

entities = ["PER", "ORG", "LOC", "MISC"]

print("\n" + "="*56)
print(f"{'Entity':<10} | {'Precision':<10} | {'Recall':<10} | {'F1':<10} | {'Support'}")
print("-" * 56)
for ent in entities:
    if ent in report_dict:
        p = report_dict[ent]["precision"]
        r = report_dict[ent]["recall"]
        f1 = report_dict[ent]["f1-score"]
        sup = report_dict[ent]["support"]
        print(f"{ent:<10} | {p:<10.4f} | {r:<10.4f} | {f1:<10.4f} | {sup}")
print("="*56)


Entity     | Precision  | Recall     | F1         | Support
--------------------------------------------------------
PER        | 0.9722     | 0.9459     | 0.9589     | 111
ORG        | 0.1667     | 0.5000     | 0.2500     | 2
LOC        | 0.7238     | 0.9268     | 0.8128     | 82
MISC       | 0.6429     | 0.4091     | 0.5000     | 22


# 22. Experiment 5 — Entity Length

In [13]:
from seqeval.metrics.sequence_labeling import get_entities

def evaluate_by_entity_length(y_true_sents, y_pred_sents):
    buckets = ["1", "2", "3", "4+"]
    gold_by_len = {b: set() for b in buckets}
    pred_by_len = {b: set() for b in buckets}

    def get_bucket(length):
        if length == 1:
            return "1"
        elif length == 2:
            return "2"
        elif length == 3:
            return "3"
        else:
            return "4+"

    for sent_idx, (gold_seq, pred_seq) in enumerate(zip(y_true_sents, y_pred_sents)):
        for ent_type, start, end in get_entities(gold_seq):
            length = end - start + 1
            gold_by_len[get_bucket(length)].add((sent_idx, ent_type, start, end))

        for ent_type, start, end in get_entities(pred_seq):
            length = end - start + 1
            pred_by_len[get_bucket(length)].add((sent_idx, ent_type, start, end))

    print("\n" + "="*68)
    print(f"{'Entity length':<15} | {'Precision':<10} | {'Recall':<10} | {'F1':<10} | {'Gold Count'}")
    print("-" * 68)

    for b in buckets:
        g_set = gold_by_len[b]
        p_set = pred_by_len[b]
        tp = len(g_set & p_set)
        fp = len(p_set - g_set)
        fn = len(g_set - p_set)

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

        print(f"{b:<15} | {prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f} | {len(g_set)}")
    print("="*68)

evaluate_by_entity_length(y_true_sents, y_pred_tf_sents)


Entity length   | Precision  | Recall     | F1         | Gold Count
--------------------------------------------------------------------
1               | 0.7029     | 0.9327     | 0.8017     | 104
2               | 0.9892     | 0.8598     | 0.9200     | 107
3               | 1.0000     | 0.3333     | 0.5000     | 6
4+              | 0.0000     | 0.0000     | 0.0000     | 0


# 23. Experiment 6 — Error Analysis

In [15]:
from seqeval.metrics.sequence_labeling import get_entities

fp_cases = []
fn_cases = []
boundary_cases = []

for sent_idx, (row, gold_seq, pred_seq) in enumerate(zip(test_data, y_true_sents, y_pred_tf_sents)):
    tokens = row["tokens"]
    sent_str = " ".join(tokens)

    gold_ents = get_entities(gold_seq)
    pred_ents = get_entities(pred_seq)

    gold_set = set(gold_ents)
    pred_set = set(pred_ents)

    matched_gold = set()
    matched_pred = set()
    for g_type, g_s, g_e in gold_ents:
        for p_type, p_s, p_e in pred_ents:
            if (g_type, g_s, g_e) == (p_type, p_s, p_e):
                continue
            if max(g_s, p_s) <= min(g_e, p_e):
                g_span = " ".join(tokens[g_s:g_e+1])
                p_span = " ".join(tokens[p_s:p_e+1])
                if g_type == p_type:
                    boundary_cases.append((sent_str, f"{g_span} ({g_type})", f"{p_span} ({p_type})"))
                    matched_gold.add((g_type, g_s, g_e))
                    matched_pred.add((p_type, p_s, p_e))
                else:
                    fp_cases.append((sent_str, f"{g_span} ({g_type})", f"{p_span} ({p_type})", "wrong entity type"))
                    matched_gold.add((g_type, g_s, g_e))
                    matched_pred.add((p_type, p_s, p_e))

    #False Positives thuần túy
    for p_type, p_s, p_e in pred_ents:
        if (p_type, p_s, p_e) not in gold_set and (p_type, p_s, p_e) not in matched_pred:
            p_span = " ".join(tokens[p_s:p_e+1])
            fp_cases.append((sent_str, f"{p_span} (O)", f"{p_span} ({p_type})", "spurious entity"))

    #Tìm False Negatives thuần túy
    for g_type, g_s, g_e in gold_ents:
        if (g_type, g_s, g_e) not in pred_set and (g_type, g_s, g_e) not in matched_gold:
            g_span = " ".join(tokens[g_s:g_e+1])
            fn_cases.append((sent_str, f"{g_span} ({g_type})", f"{g_span} (O)", "missing entity"))

print(f"\nTổng hợp lỗi tìm thấy: {len(fp_cases)} FP | {len(fn_cases)} FN | {len(boundary_cases)} Boundary Errors")

print("\n=== TOP 5 FALSE POSITIVES (FP) ===")
for i, (txt, g, p, err) in enumerate(fp_cases[:5], 1):
    print(f"{i}. Text: {txt}\n   Gold: {g} | Pred: {p} | Type: {err}")

print("\n=== TOP 5 FALSE NEGATIVES (FN) ===")
for i, (txt, g, p, err) in enumerate(fn_cases[:5], 1):
    print(f"{i}. Text: {txt}\n   Gold: {g} | Pred: {p} | Type: {err}")

print("\n=== TOP 5 BOUNDARY ERRORS ===")
for i, (txt, g, p) in enumerate(boundary_cases[:5], 1):
    print(f"{i}. Text: {txt}\n   Gold: {g} | Pred: {p} | Type: wrong boundary")


Tổng hợp lỗi tìm thấy: 18 FP | 1 FN | 24 Boundary Errors

=== TOP 5 FALSE POSITIVES (FP) ===
1. Text: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
   Gold: CHINA (PER) | Pred: CHINA (LOC) | Type: wrong entity type
2. Text: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
   Gold: Asian Cup (MISC) | Pred: Asian (LOC) | Type: wrong entity type
3. Text: The former Soviet republic was playing in an Asian Cup finals tie for the first time .
   Gold: Asian Cup (MISC) | Pred: Asian (LOC) | Type: wrong entity type
4. Text: The former Soviet republic was playing in an Asian Cup finals tie for the first time .
   Gold: former (O) | Pred: former (LOC) | Type: spurious entity
5. Text: The former Soviet republic was playing in an Asian Cup finals tie for the first time .
   Gold: republic (O) | Pred: republic (LOC) | Type: spurious entity

=== TOP 5 FALSE NEGATIVES (FN) ===
1. Text: FREESTYLE SKIING-WORLD C

# 24. Context Analysis

In [19]:
from collections import defaultdict

word_contexts = defaultdict(lambda: defaultdict(list))

for row, gold_seq, pred_seq in zip(test_data, y_true_sents, y_pred_tf_sents):
    tokens = row["tokens"]
    sent_str = " ".join(tokens)

    for tok, g_lbl, p_lbl in zip(tokens, gold_seq, pred_seq):
        if tok.isalpha() and len(tok) > 2:
            ent_type = g_lbl.split("-")[-1] if "-" in g_lbl else "O"
            word_contexts[tok.lower()][ent_type].append((tok, g_lbl, p_lbl, sent_str))

polysemous_words = {
    w: types for w, types in word_contexts.items() if len(types) >= 2
}

print(f"Tìm thấy {len(polysemous_words)} từ có cùng mặt chữ nhưng mang nhãn khác nhau.\n")

count = 0
for w, types_dict in sorted(polysemous_words.items(), key=lambda x: len(x[1]), reverse=True):
    if set(types_dict.keys()) == {"O"}:
        continue
    count += 1
    print(f"Ví dụ {count}: Từ bề mặt (Surface form) = '{w}'")
    for ent_type, examples in list(types_dict.items())[:3]:
        orig_tok, g_lbl, p_lbl, sent = examples[0]
        print(f"  • Loại nhãn: {ent_type:<5} (Gold: {g_lbl:<6} | Pred: {p_lbl:<6}) -> Câu: \"{sent}\"")
    print()
    if count >= 5:
        break

Tìm thấy 2 từ có cùng mặt chữ nhưng mang nhãn khác nhau.

Ví dụ 1: Từ bề mặt (Surface form) = 'china'
  • Loại nhãn: PER   (Gold: B-PER  | Pred: B-LOC ) -> Câu: "SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT ."
  • Loại nhãn: LOC   (Gold: B-LOC  | Pred: B-LOC ) -> Câu: "But China saw their luck desert them in the second match of the group , crashing to a surprise 2-0 defeat to newcomers Uzbekistan ."

Ví dụ 2: Từ bề mặt (Surface form) = 'world'
  • Loại nhãn: MISC  (Gold: B-MISC | Pred: B-MISC) -> Câu: "Japan , co-hosts of the World Cup in 2002 and ranked 20th in the world by FIFA , are favourites to regain their title here ."
  • Loại nhãn: O     (Gold: O      | Pred: O     ) -> Câu: "Japan , co-hosts of the World Cup in 2002 and ranked 20th in the world by FIFA , are favourites to regain their title here ."



# 25. Application — Information Extraction

In [20]:
import json
import torch

TYPE_MAPPING = {
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "LOC": "LOCATION",
    "MISC": "MISCELLANEOUS"
}

def extract_structured_info(raw_text, model, tokenizer, id2label, device):
    """
    Pipeline IE hoàn chỉnh:
    Raw text -> Pre-tokenization -> Subword Tokenization -> NER Model -> Span Merging -> JSON
    """
    model.eval()

    words = raw_text.replace(".", " .").replace(",", " ,").split()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    word_ids = encoded.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    word_preds = []
    prev_word_idx = None
    for pred_id, w_idx in zip(pred_ids, word_ids):
        if w_idx is not None and w_idx != prev_word_idx:
            word_preds.append(id2label[pred_id])
        prev_word_idx = w_idx

    structured_data = {
        "PERSON": [],
        "ORGANIZATION": [],
        "LOCATION": [],
        "MISCELLANEOUS": []
    }

    current_entity_tokens = []
    current_entity_type = None

    for word, tag in zip(words, word_preds):
        if tag.startswith("B-"):
            if current_entity_tokens and current_entity_type:
                entity_str = " ".join(current_entity_tokens)
                if entity_str not in structured_data[current_entity_type]:
                    structured_data[current_entity_type].append(entity_str)

            short_type = tag.split("-")[1]
            current_entity_type = TYPE_MAPPING.get(short_type, short_type)
            current_entity_tokens = [word]

        elif tag.startswith("I-") and current_entity_type == TYPE_MAPPING.get(tag.split("-")[1]):
            current_entity_tokens.append(word)

        else:
            if current_entity_tokens and current_entity_type:
                entity_str = " ".join(current_entity_tokens)
                if entity_str not in structured_data[current_entity_type]:
                    structured_data[current_entity_type].append(entity_str)
            current_entity_tokens = []
            current_entity_type = None

    if current_entity_tokens and current_entity_type:
        entity_str = " ".join(current_entity_tokens)
        if entity_str not in structured_data[current_entity_type]:
            structured_data[current_entity_type].append(entity_str)

    return {k: v for k, v in structured_data.items() if len(v) > 0}

sample_articles = [
    "Barack Obama visited Paris.",
    "Elon Musk founded SpaceX in California and later acquired Twitter.",
    "Tim Cook announced that Apple will open a new research center in Tokyo and London."
]

for idx, text in enumerate(sample_articles, 1):
    extracted_json = extract_structured_info(text, model, tokenizer, id2label, device)
    print(f"\n[Văn bản thô {idx}]: \"{text}\"")
    print("[Dữ liệu có cấu trúc (JSON)]:")
    print(json.dumps(extracted_json, indent=2, ensure_ascii=False))


[Văn bản thô 1]: "Barack Obama visited Paris."
[Dữ liệu có cấu trúc (JSON)]:
{
  "PERSON": [
    "Barack Obama"
  ],
  "LOCATION": [
    "Paris"
  ]
}

[Văn bản thô 2]: "Elon Musk founded SpaceX in California and later acquired Twitter."
[Dữ liệu có cấu trúc (JSON)]:
{
  "PERSON": [
    "Elon Musk"
  ],
  "ORGANIZATION": [
    "SpaceX",
    "Twitter"
  ],
  "LOCATION": [
    "California"
  ]
}

[Văn bản thô 3]: "Tim Cook announced that Apple will open a new research center in Tokyo and London."
[Dữ liệu có cấu trúc (JSON)]:
{
  "PERSON": [
    "Tim Cook"
  ],
  "ORGANIZATION": [
    "Apple"
  ],
  "LOCATION": [
    "Tokyo",
    "London"
  ]
}


# 27. Evaluation của Information Extraction

In [21]:
import json
import torch

OPTION_A_MAP = {
    "PER": "PERSON",
    "ORG": "ORG",
    "LOC": "LOCATION"
}

def run_news_ie_pipeline(raw_text, model, tokenizer, id2label, device):
    model.eval()
    words = raw_text.replace(".", " .").replace(",", " ,").replace("'s", " 's").split()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    word_ids = encoded.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    word_preds = []
    prev_w = None
    for p_id, w_id in zip(pred_ids, word_ids):
        if w_id is not None and w_id != prev_w:
            word_preds.append(id2label[p_id])
        prev_w = w_id

    extracted = {"PERSON": [], "ORG": [], "LOCATION": []}
    curr_tokens = []
    curr_type = None

    for word, tag in zip(words, word_preds):
        if tag.startswith("B-"):
            if curr_tokens and curr_type in extracted:
                ent_str = " ".join(curr_tokens)
                if ent_str not in extracted[curr_type]:
                    extracted[curr_type].append(ent_str)
            short_t = tag.split("-")[1]
            curr_type = OPTION_A_MAP.get(short_t, None)
            curr_tokens = [word] if curr_type else []
        elif tag.startswith("I-") and curr_type == OPTION_A_MAP.get(tag.split("-")[1], None):
            curr_tokens.append(word)
        else:
            if curr_tokens and curr_type in extracted:
                ent_str = " ".join(curr_tokens)
                if ent_str not in extracted[curr_type]:
                    extracted[curr_type].append(ent_str)
            curr_tokens = []
            curr_type = None

    if curr_tokens and curr_type in extracted:
        ent_str = " ".join(curr_tokens)
        if ent_str not in extracted[curr_type]:
            extracted[curr_type].append(ent_str)

    return extracted

news_eval_30 = [
    ("Barack Obama visited Paris yesterday.", {"PERSON": ["Barack Obama"], "ORG": [], "LOCATION": ["Paris"]}),
    ("Elon Musk founded SpaceX in California.", {"PERSON": ["Elon Musk"], "ORG": ["SpaceX"], "LOCATION": ["California"]}),
    ("Tim Cook announced new Apple devices in Cupertino.", {"PERSON": ["Tim Cook"], "ORG": ["Apple"], "LOCATION": ["Cupertino"]}),
    ("Microsoft acquired GitHub for billions of dollars.", {"PERSON": [], "ORG": ["Microsoft", "GitHub"], "LOCATION": []}),
    ("Joe Biden met Emmanuel Macron in Washington.", {"PERSON": ["Joe Biden", "Emmanuel Macron"], "ORG": [], "LOCATION": ["Washington"]}),
    ("Toyota opened a manufacturing plant in Tokyo.", {"PERSON": [], "ORG": ["Toyota"], "LOCATION": ["Tokyo"]}),
    ("Sundar Pichai leads Google from Mountain View.", {"PERSON": ["Sundar Pichai"], "ORG": ["Google"], "LOCATION": ["Mountain View"]}),
    ("The United Nations held a summit in Geneva.", {"PERSON": [], "ORG": ["United Nations"], "LOCATION": ["Geneva"]}),
    ("Angela Merkel traveled to Berlin for talks.", {"PERSON": ["Angela Merkel"], "ORG": [], "LOCATION": ["Berlin"]}),
    ("Samsung Electronics reported strong profits in Seoul.", {"PERSON": [], "ORG": ["Samsung Electronics"], "LOCATION": ["Seoul"]}),
    ("Jeff Bezos stepped down as chief executive of Amazon.", {"PERSON": ["Jeff Bezos"], "ORG": ["Amazon"], "LOCATION": []}),
    ("The European Central Bank is based in Frankfurt.", {"PERSON": [], "ORG": ["European Central Bank"], "LOCATION": ["Frankfurt"]}),
    ("Cristiano Ronaldo joined Al Nassr in Riyadh.", {"PERSON": ["Cristiano Ronaldo"], "ORG": ["Al Nassr"], "LOCATION": ["Riyadh"]}),
    ("Mark Zuckerberg testified before Congress in Washington.", {"PERSON": ["Mark Zuckerberg"], "ORG": ["Congress"], "LOCATION": ["Washington"]}),
    ("Volkswagen expanded operations across Germany and France.", {"PERSON": [], "ORG": ["Volkswagen"], "LOCATION": ["Germany", "France"]}),
    ("Satya Nadella spoke at the World Economic Forum in Davos.", {"PERSON": ["Satya Nadella"], "ORG": ["World Economic Forum"], "LOCATION": ["Davos"]}),
    ("Boeing delivered new aircraft to Lufthansa.", {"PERSON": [], "ORG": ["Boeing", "Lufthansa"], "LOCATION": []}),
    ("Lionel Messi moved to Inter Miami in Florida.", {"PERSON": ["Lionel Messi"], "ORG": ["Inter Miami"], "LOCATION": ["Florida"]}),
    ("The World Health Organization issued guidelines from Switzerland.", {"PERSON": [], "ORG": ["World Health Organization"], "LOCATION": ["Switzerland"]}),
    ("Jensen Huang unveiled new Nvidia chips in Taiwan.", {"PERSON": ["Jensen Huang"], "ORG": ["Nvidia"], "LOCATION": ["Taiwan"]}),
    ("Rishi Sunak addressed Parliament in London.", {"PERSON": ["Rishi Sunak"], "ORG": ["Parliament"], "LOCATION": ["London"]}),
    ("Intel invested heavily in Arizona and Ohio.", {"PERSON": [], "ORG": ["Intel"], "LOCATION": ["Arizona", "Ohio"]}),
    ("Christine Lagarde discussed inflation in Brussels.", {"PERSON": ["Christine Lagarde"], "ORG": [], "LOCATION": ["Brussels"]}),
    ("Sony Pictures released a blockbuster film in Los Angeles.", {"PERSON": [], "ORG": ["Sony Pictures"], "LOCATION": ["Los Angeles"]}),
    ("Antonio Guterres visited New York for climate negotiations.", {"PERSON": ["Antonio Guterres"], "ORG": [], "LOCATION": ["New York"]}),
    ("Airbus signed a major contract with Emirates in Dubai.", {"PERSON": [], "ORG": ["Airbus", "Emirates"], "LOCATION": ["Dubai"]}),
    ("Novak Djokovic won another championship in Melbourne.", {"PERSON": ["Novak Djokovic"], "ORG": [], "LOCATION": ["Melbourne"]}),
    ("Goldman Sachs analyzed markets in Wall Street and London.", {"PERSON": [], "ORG": ["Goldman Sachs"], "LOCATION": ["Wall Street", "London"]}),
    ("Justin Trudeau met reporters in Ottawa.", {"PERSON": ["Justin Trudeau"], "ORG": [], "LOCATION": ["Ottawa"]}),
    ("Pfizer collaborated with BioNTech on medical research.", {"PERSON": [], "ORG": ["Pfizer", "BioNTech"], "LOCATION": []})
]

tp_total, fp_total, fn_total = 0, 0, 0
per_type_stats = {k: {"tp": 0, "fp": 0, "fn": 0} for k in ["PERSON", "ORG", "LOCATION"]}

print("\n[Minh họa 3 kết quả trích xuất JSON đầu tiên]:")
for idx, (sentence, gold_dict) in enumerate(news_eval_30):
    pred_dict = run_news_ie_pipeline(sentence, model, tokenizer, id2label, device)

    if idx < 3:
        print(f"• Câu {idx+1}: \"{sentence}\"")
        print(f"  -> Extracted JSON: {json.dumps(pred_dict)}")

    for ent_type in ["PERSON", "ORG", "LOCATION"]:
        gold_set = set(gold_dict.get(ent_type, []))
        pred_set = set(pred_dict.get(ent_type, []))

        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

        tp_total += tp
        fp_total += fp
        fn_total += fn

        per_type_stats[ent_type]["tp"] += tp
        per_type_stats[ent_type]["fp"] += fp
        per_type_stats[ent_type]["fn"] += fn

precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0.0
recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "="*62)
print("KẾT QUẢ ĐÁNH GIÁ INFORMATION EXTRACTION (30 SENTENCES - OPTION A)")
print("-" * 62)
print(f"{'Entity Schema':<16} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Precision':<9} | {'Recall':<8} | {'F1'}")
print("-" * 62)

for ent_type in ["PERSON", "ORG", "LOCATION"]:
    s = per_type_stats[ent_type]
    p_t = s["tp"] / (s["tp"] + s["fp"]) if (s["tp"] + s["fp"]) > 0 else 0.0
    r_t = s["tp"] / (s["tp"] + s["fn"]) if (s["tp"] + s["fn"]) > 0 else 0.0
    f1_t = (2 * p_t * r_t) / (p_t + r_t) if (p_t + r_t) > 0 else 0.0
    print(f"{ent_type:<16} | {s['tp']:<5} | {s['fp']:<5} | {s['fn']:<5} | {p_t:<9.4f} | {r_t:<8.4f} | {f1_t:.4f}")

print("-" * 62)
print(f"{'OVERALL (Micro)':<16} | {tp_total:<5} | {fp_total:<5} | {fn_total:<5} | {precision:<9.4f} | {recall:<8.4f} | {f1:.4f}")
print("="*62)


[Minh họa 3 kết quả trích xuất JSON đầu tiên]:
• Câu 1: "Barack Obama visited Paris yesterday."
  -> Extracted JSON: {"PERSON": ["Barack Obama"], "ORG": [], "LOCATION": ["Paris"]}
• Câu 2: "Elon Musk founded SpaceX in California."
  -> Extracted JSON: {"PERSON": ["Elon Musk"], "ORG": ["SpaceX"], "LOCATION": ["California"]}
• Câu 3: "Tim Cook announced new Apple devices in Cupertino."
  -> Extracted JSON: {"PERSON": ["Tim Cook"], "ORG": [], "LOCATION": ["Cupertino"]}

KẾT QUẢ ĐÁNH GIÁ INFORMATION EXTRACTION (30 SENTENCES - OPTION A)
--------------------------------------------------------------
Entity Schema    | TP    | FP    | FN    | Precision | Recall   | F1
--------------------------------------------------------------
PERSON           | 18    | 0     | 0     | 1.0000    | 1.0000   | 1.0000
ORG              | 20    | 6     | 7     | 0.7692    | 0.7407   | 0.7547
LOCATION         | 25    | 8     | 4     | 0.7576    | 0.8621   | 0.8065
-----------------------------------------------